In [1]:
import os
import cv2
import numpy as np
import torch
import json
import math
from glob import glob
from collections import defaultdict

In [2]:
# ===== ENVIRONMENT SETUP =====
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install detectron2==0.6 opencv-python fastdtw matplotlib tqdm
!pip install gdown

import os, zipfile, torch


Looking in indexes: https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement detectron2==0.6 (from versions: none)
ERROR: No matching distribution found for detectron2==0.6


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Path to your dataset in Drive
drive_zip_path = '/content/drive/MyDrive/archive.zip'
extract_path = '/content/archive'

# Extract dataset if not already extracted
if not os.path.exists(extract_path):
    # Create the target directory if it doesn't exist
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(drive_zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path) # Extract to the new directory
    print("✅ Dataset extracted to:", extract_path)
else:
    print("✅ Dataset already extracted:", extract_path)

Mounted at /content/drive
✅ Dataset extracted to: /content/archive


In [4]:
# --------- CONFIG ----------
import cv2, numpy as np, json
import torch
from glob import glob
from collections import defaultdict
import matplotlib.pyplot as plt
from tqdm import tqdm

ARCHIVE_PATH = '/content/archive'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

DETECTRON2_CONFIG = 'COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml'
DETECTRON2_WEIGHTS = 'detectron2://COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x/137849621/model_final_a6e10b.pkl'

COCO_KP_NAMES = [
    'nose','left_eye','right_eye','left_ear','right_ear',
    'left_shoulder','right_shoulder','left_elbow','right_elbow','left_wrist','right_wrist',
    'left_hip','right_hip','left_knee','right_knee','left_ankle','right_ankle'
]

ANGLE_JOINTS = {
    'left_elbow': ('left_shoulder','left_elbow','left_wrist'),
    'right_elbow': ('right_shoulder','right_elbow','right_wrist'),
    'left_shoulder': ('left_elbow','left_shoulder','left_hip'),
    'right_shoulder': ('right_elbow','right_shoulder','right_hip'),
    'left_knee': ('left_hip','left_knee','left_ankle'),
    'right_knee': ('right_hip','right_knee','right_ankle'),
    'left_hip': ('left_shoulder','left_hip','left_knee'),
    'right_hip': ('right_shoulder','right_hip','right_knee')
}


In [5]:
# Install detectron2 using a different method
!python -m pip install 'detectron2@git+https://github.com/facebookresearch/detectron2.git'

  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-install-c08kkgbh/detectron2_e91bab3c09584bd5b761adac9e044c8e
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-install-c08kkgbh/detectron2_e91bab3c09584bd5b761adac9e044c8e
  Resolved https://github.com/facebookresearch/detectron2.git to commit a9c0821a12ad353fb2a96f019515990d5460c5ac
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 56.1 MB/s eta 0:00:00
  Created wheel for detectron2: filename=detectron2-0.6-cp312-cp312-linux_x86_64.whl size=6733336 sha256=5f13ae09b13e9d3eefd7be74470952a80d4dc1df8681e9ebb99a5586199b5cb9
  Store

In [6]:
# ===== DETECTRON2 SETUP =====
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo

def init_detectron2():
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(DETECTRON2_CONFIG))
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(DETECTRON2_CONFIG)
    cfg.MODEL.DEVICE = DEVICE
    return DefaultPredictor(cfg)

predictor = init_detectron2()

def run_detectron2_on_frame(frame):
    outputs = predictor(frame)
    keypoints = outputs["instances"].pred_keypoints
    if len(keypoints) == 0:
        return None
    keypoints = keypoints[0].cpu().numpy()  # take first person
    return keypoints[:, :2]  # (x,y)


model_final_a6e10b.pkl: 237MB [00:01, 168MB/s]                           


In [7]:
# ===== UTILITIES =====
def read_video_frames(video_path, resize=(640, 360), max_frames=200):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret or len(frames) >= max_frames:
            break
        frame = cv2.resize(frame, resize)
        frames.append(frame)
    cap.release()
    return frames

def angle_between_points(a, b, c):
    ba = a - b
    bc = c - b
    cosang = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return np.degrees(np.arccos(np.clip(cosang, -1.0, 1.0)))

def compute_angles_from_keypoints(kps):
    angles = {}
    for jname, (a, b, c) in ANGLE_JOINTS.items():
        try:
            a_pt, b_pt, c_pt = np.array(kps[COCO_KP_NAMES.index(a)]), np.array(kps[COCO_KP_NAMES.index(b)]), np.array(kps[COCO_KP_NAMES.index(c)])
            angles[jname] = angle_between_points(a_pt, b_pt, c_pt)
        except:
            angles[jname] = None
    return angles


In [8]:
import os

print("Contents of /content/:")
print(os.listdir('/content/'))

print("\nContents of /content/archive/ (if it exists):")
if os.path.exists('/content/archive/'):
    print(os.listdir('/content/archive/'))
else:
    print("'/content/archive/' does not exist.")

Contents of /content/:
['.config', 'drive', 'archive', 'sample_data']

Contents of /content/archive/ (if it exists):
['hip thrust', 'lateral raise', 'pull Up', 'bench press', 'tricep Pushdown', 'chest fly machine', 'decline bench press', 'deadlift', 'leg raises', 'barbell biceps curl', 'russian twist', 'romanian deadlift', 'hammer curl', 'squat', 'plank', 'shoulder press', 'leg extension', 'push-up', 't bar row', 'incline bench press', 'tricep dips', 'lat pulldown']


In [9]:
# ===== COMPUTE THRESHOLDS FROM DATASET =====
def compute_thresholds_from_dataset(base_path=ARCHIVE_PATH, k=2.0, resize=(640,360)):
    classes = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
    thresholds = {}
    for cls in classes:
        joint_angles = defaultdict(list)
        vids = glob(os.path.join(base_path, cls, '*.mp4'))
        print(f"🔹 Processing {cls} with {len(vids)} videos")
        for v in tqdm(vids):
            frames = read_video_frames(v, resize=resize, max_frames=200)
            for f in frames[::5]:  # sample every 5th frame
                kps = run_detectron2_on_frame(f)
                if kps is None: continue
                angs = compute_angles_from_keypoints(kps)
                for j, val in angs.items():
                    if val is not None:
                        joint_angles[j].append(val)
        cls_th = {}
        for j, vals in joint_angles.items():
            if len(vals) < 20: continue
            mean, std = np.mean(vals), np.std(vals)
            cls_th[j] = {'min': float(mean - k*std), 'max': float(mean + k*std)}
        thresholds[cls] = cls_th
    os.makedirs('models', exist_ok=True)
    with open('models/thresholds.json', 'w') as f:
        json.dump(thresholds, f, indent=2)
    print("✅ Thresholds saved to models/thresholds.json")
    return thresholds

EXERCISE_THRESHOLDS = compute_thresholds_from_dataset(base_path=ARCHIVE_PATH)

🔹 Processing hip thrust with 14 videos


  0%|          | 0/14 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
100%|██████████| 14/14 [01:21<00:00,  5.80s/it]


🔹 Processing lateral raise with 31 videos


100%|██████████| 31/31 [02:42<00:00,  5.24s/it]


🔹 Processing pull Up with 26 videos


...

🔹 Processing tricep dips with 16 videos


100%|██████████| 16/16 [01:39<00:00,  6.21s/it]


🔹 Processing lat pulldown with 51 videos


100%|██████████| 51/51 [03:28<00:00,  4.08s/it]

✅ Thresholds saved to models/thresholds.json


In [12]:
# ===== CHECK NEW VIDEO AGAINST THRESHOLDS =====
def check_video(video_path, thresholds, resize=(640,360)):
    frames = read_video_frames(video_path, resize=resize)
    results = []
    for f in tqdm(frames[::5]):  # sample every 5th frame
        kps = run_detectron2_on_frame(f)
        if kps is None:
            continue
        angs = compute_angles_from_keypoints(kps)
        frame_eval = {}
        for joint, val in angs.items():
            if joint not in thresholds:
                continue
            t = thresholds[joint]
            if val < t['min']:
                status = 'under'
            elif val > t['max']:
                status = 'over'
            else:
                status = 'ok'
            frame_eval[joint] = {'angle': val, 'status': status}
        results.append(frame_eval)
    return results


# ===== TEST SAMPLE VIDEO LOCATED IN /content/ =====
sample_video = '/content/4265287-uhd_3840_2160_30fps.mp4'   # 👈 your uploaded video path
exercise_name = 'squat'                   # must match dataset folder name

results = check_video(sample_video, EXERCISE_THRESHOLDS[exercise_name])
print(f"✅ Checked {sample_video}, got {len(results)} frames analyzed.")

# ===== OPTIONAL: Summary output =====
import pandas as pd
df = []
for fidx, frame_eval in enumerate(results):
    for joint, info in frame_eval.items():
        df.append([fidx, joint, info['angle'], info['status']])

df = pd.DataFrame(df, columns=['frame','joint','angle','status'])
print("\n🔹 First few results:")
print(df.head(15))
print("\n🔹 Status summary:")
print(df['status'].value_counts())

# (Optional) save results
df.to_csv('/content/form_analysis_results.csv', index=False)
print("\n📁 Results saved to /content/form_analysis_results.csv")


100%|██████████| 40/40 [00:05<00:00,  6.76it/s]


✅ Checked /content/4265287-uhd_3840_2160_30fps.mp4, got 40 frames analyzed.

🔹 First few results:
    frame           joint       angle status
0       0      left_elbow  177.533188   over
1       0     right_elbow  174.719574     ok
2       0   left_shoulder   13.012929     ok
3       0  right_shoulder   18.582232     ok
4       0       left_knee  175.241302     ok
5       0      right_knee  176.986160     ok
6       0        left_hip  162.715744     ok
7       0       right_hip  167.931168     ok
8       1      left_elbow  171.860321   over
9       1     right_elbow  169.694748     ok
10      1   left_shoulder   19.919924     ok
11      1  right_shoulder   22.871780     ok
12      1       left_knee  175.338501     ok
13      1      right_knee  177.445633     ok
14      1        left_hip  164.467758     ok

🔹 Status summary:
status
ok      311
over      9
Name: count, dtype: int64

📁 Results saved to /content/form_analysis_results.csv


In [13]:
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

# ===== VISUAL ANALYSIS FUNCTION =====
def analyze_and_visualize_video(video_path, thresholds, exercise_name, resize=(640,360), output_path='/content/analysis_output.mp4'):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width, height = int(resize[0]), int(resize[1])

    # Set up video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_results = []
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"🔹 Analyzing {video_path} ({frame_count} frames)...")

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (width, height))
        frame_idx += 1
        if frame_idx % 5 != 0:
            continue  # process every 5th frame for speed

        # Run pose detection
        kps = run_detectron2_on_frame(frame)
        if kps is None:
            out.write(frame)
            continue

        angs = compute_angles_from_keypoints(kps)
        frame_eval = {}

        # Visualize skeleton
        for j, (x, y) in enumerate(kps):
            name = COCO_KP_NAMES[j]
            cv2.circle(frame, (int(x), int(y)), 4, (255,255,255), -1)

        # Evaluate and color joints
        for joint, val in angs.items():
            if joint not in thresholds:
                continue
            t = thresholds[joint]
            if val < t['min']:
                color, status = (0, 0, 255), 'under'  # red
            elif val > t['max']:
                color, status = (0, 0, 255), 'over'   # red
            else:
                color, status = (0, 255, 0), 'ok'     # green
            frame_eval[joint] = {'angle': val, 'status': status}

            # Draw joint label
            b_joint = ANGLE_JOINTS[joint][1]
            bx, by = kps[COCO_KP_NAMES.index(b_joint)]
            cv2.putText(frame, joint.split('_')[1], (int(bx), int(by)-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            cv2.circle(frame, (int(bx), int(by)), 6, color, -1)

        frame_results.append(frame_eval)

        # Add exercise name overlay
        cv2.putText(frame, f"Exercise: {exercise_name}", (20, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        out.write(frame)

    cap.release()
    out.release()
    print(f"✅ Analysis complete. Saved visualization to {output_path}")
    return frame_results

# ===== RUN ANALYSIS ON SAMPLE VIDEO =====
sample_video = '/content/4265287-uhd_3840_2160_30fps.mp4'   # your uploaded video
exercise_name = 'squat'

results = analyze_and_visualize_video(
    sample_video,
    EXERCISE_THRESHOLDS[exercise_name],
    exercise_name,
    output_path='/content/squat_analysis.mp4'
)

# ===== OPTIONAL: VIEW SUMMARY =====
df = []
for fidx, frame_eval in enumerate(results):
    for joint, info in frame_eval.items():
        df.append([fidx, joint, info['angle'], info['status']])

df = pd.DataFrame(df, columns=['frame','joint','angle','status'])
print("\n🔹 Status summary:")
print(df['status'].value_counts())

print("\n📹 Visualization video saved at: /content/squat_analysis.mp4")




🔹 Analyzing /content/4265287-uhd_3840_2160_30fps.mp4 (361 frames)...
✅ Analysis complete. Saved visualization to /content/squat_analysis.mp4

🔹 Status summary:
status
ok      565
over     11
Name: count, dtype: int64

📹 Visualization video saved at: /content/squat_analysis.mp4


In [30]:
import cv2
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from IPython.display import HTML
from base64 import b64encode

def analyze_and_visualize_video(video_path, thresholds, exercise_name, resize=(640,360), output_path='/content/analysis_output.mp4'):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width, height = int(resize[0]), int(resize[1])

    # Use mp4v (universally supported in Colab)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    frame_results = []
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"🔹 Analyzing {video_path} ({frame_count} frames at {fps:.2f} fps)...")

    skip = 5  # process every 5th frame
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1
        if frame_idx % skip != 0:
            continue

        frame = cv2.resize(frame, (width, height))
        kps = run_detectron2_on_frame(frame)
        if kps is None:
            for _ in range(skip):
                out.write(frame)
            continue

        angs = compute_angles_from_keypoints(kps)
        frame_eval = {}

        # Draw keypoints
        for j, (x, y) in enumerate(kps):
            cv2.circle(frame, (int(x), int(y)), 3, (255, 255, 255), -1)

        # Evaluate and annotate joints
        for joint, val in angs.items():
            if joint not in thresholds:
                continue
            t = thresholds[joint]
            if val < t['min']:
                color, status = (0, 0, 255), 'under'
            elif val > t['max']:
                color, status = (0, 0, 255), 'over'
            else:
                color, status = (0, 255, 0), 'ok'

            frame_eval[joint] = {'angle': val, 'status': status}

            # Mark and label
            b_joint = ANGLE_JOINTS[joint][1]
            bx, by = kps[COCO_KP_NAMES.index(b_joint)]
            cv2.circle(frame, (int(bx), int(by)), 6, color, -1)
            cv2.putText(frame, f"{joint.split('_')[1]} {int(val)}°", (int(bx)+5, int(by)-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

        # Overlay exercise info
        cv2.putText(frame, f"Exercise: {exercise_name}", (20, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        frame_results.append(frame_eval)

        # Write this frame multiple times to maintain playback speed
        for _ in range(skip):
            out.write(frame)

    cap.release()
    out.release()
    # cv2.destroyAllWindows() # Removed this line
    print(f"✅ Analysis complete. Saved video at: {output_path}")
    return frame_results


# ===== RUN FIXED ANALYSIS =====
sample_video = '/content/squat sample.mp4'
exercise_name = 'squat'

results = analyze_and_visualize_video(
    sample_video,
    EXERCISE_THRESHOLDS[exercise_name],
    exercise_name,
    output_path='/content/squat_analysis1.mp4'
)



🔹 Analyzing /content/squat sample.mp4 (361 frames at 29.97 fps)...
✅ Analysis complete. Saved video at: /content/squat_analysis1.mp4
